In [2]:
import pandas as pd
import requests
import json


In [3]:
# Configuration
BASE_URL = "https://cms.correlaid.org/graphql"

# Your GraphQL Query
query = """
query ProjectOverview($language: String = "de-DE", $status: [String] = ["published"]) {
	Projects(limit: -1, filter: { status: { _in: $status }  } ) {
		status
		project_id
		project_status
		is_internal
		end_date_predicted
		end_date
		project_types
		data_types
		Podcast {
			language
			soundcloud_link
			title
		}
		Blog_Posts {
			Blog_Posts_id(filter: { status: { _in: $status } }) {
				id
				translations {
					languages_code {
						code
					}
					title
					slug
				}
			}
		}
		Projects_Outputs(filter: { is_public: { _eq: true } }) {
			url
			output_type
		}
		Organizations {
			Organizations_id {
				sector
				translations(filter: { languages_code: { code: { _eq: $language } } }) {
					languages_code {
						code
					}
					name
				}
			}
		}
		translations(filter: { languages_code: { code: { _eq: $language } } }) {
			title
			teaser
		}
		Local_Chapters {
			Local_Chapters_id (filter: { status: { _in: $status } }){
				short_id
				translations(filter: { languages_code: { code: { _eq: $language } } }) {
					city
				}
			}
		}
	}
}
"""

In [4]:
def fetch_project_overview():
    try:
        # Directus GraphQL is usually a POST request to the /graphql endpoint
        response = requests.post(
            BASE_URL,
            json={'query': query},
            timeout=10
        )
        
        # Check for HTTP errors
        response.raise_for_status()
        
        data = response.json()
        
        if 'errors' in data:
            print("GraphQL Errors:", data['errors'])
            return None
            
        return data['data']['Projects']

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None

# Execute
projects = fetch_project_overview()

In [5]:
print(f"Successfully retrieved {len(projects)} projects.")

Successfully retrieved 38 projects.


# Normalize Directus JSON Response

In [6]:
df = pd.json_normalize(projects)

In [7]:
df

,status,project_id,project_status,is_internal,end_date_predicted,end_date,project_types,data_types,Podcast,Blog_Posts,Projects_Outputs,Organizations,translations,Local_Chapters,Podcast.language,Podcast.soundcloud_link,Podcast.title
0,published,2022-11-ATT,finished,False,2022-11-30,2022-11-20,"[exploratory, reporting, visualization]","[open, process]",NaN,[],[{'url': 'https://correlaid.github.io/trinkbru...,[{'Organizations_id': {'sector': 'environment'...,[{'title': 'Qualitätsanalyse von OpenStreetMap...,"[{'Local_Chapters_id': {'short_id': 'hamburg',...",NaN,NaN,NaN
1,published,2023-10-ISH,finished,False,2024-03-15,2024-07-01,"[modeling, ai, nlp]","[text, process]",NaN,[],[],[{'Organizations_id': {'sector': 'arts_culture...,[{'title': 'Mithilfe von KI Wirkungsmessung sk...,[],NaN,NaN,NaN
2,published,2021-04-SOS,finished,False,2021-08-15,2021-12-08,[automation],[process],NaN,[],[],[{'Organizations_id': {'sector': 'education_re...,[{'title': 'Wertvolle Zeit sparen und Fehler v...,[],NaN,NaN,NaN
3,published,2024-09-CHW,finished,False,None,2025-01-31,"[reporting, visualization, automation, impact]","[panel, survey]",NaN,[],[],[{'Organizations_id': {'sector': 'education_re...,[{'title': 'Automatisiertes Monitoring von Zie...,[],NaN,NaN,NaN
4,published,2024-06-BAB,finished,False,2024-09-20,2024-10-09,[reporting],[survey],NaN,[],[{'url': 'https://github.com/CorrelAid/worksho...,"[{'Organizations_id': {'sector': 'health', 'tr...",[{'title': 'Automatisierte Fragebogenauswertun...,[],NaN,NaN,NaN
5,published,2024-12-SIN,project_work,False,None,2025-05-31,"[reporting, visualization, automation]","[survey, process, administrative]",NaN,[],[],[{'Organizations_id': {'sector': 'education_re...,[{'title': 'Automatisiertes Qualitätsmanagemen...,[],NaN,NaN,NaN
6,published,2023-12-DEM,finished,False,None,2024-02-15,"[open_source, visualization]","[open, administrative]",NaN,[],[{'url': 'https://civic-data.de/ein-wegweiser-...,[{'Organizations_id': {'sector': 'law_advocacy...,[{'title': 'Mit Daten zu transparenterer Demok...,[],NaN,NaN,NaN
7,published,2024-04-TEN,project_work,False,None,None,[data_collection],[survey],NaN,[],[{'url': 'https://civic-data.de/output-monitor...,[{'Organizations_id': {'sector': 'arts_culture...,[{'title': 'Output Monitoring durch Entwicklun...,[],NaN,NaN,NaN
8,published,2024-03-PZA,finished,False,None,2024-11-15,"[data_collection, data_management, ai]","[open, administrative, survey]",NaN,[],[{'url': 'https://civic-data.de/kommuki-open-d...,[{'Organizations_id': {'sector': 'intermediari...,[{'title': 'Open Data aus Bürger- und Jugendbe...,[],NaN,NaN,NaN
9,published,2024-03-LEM,finished,False,None,2024-10-15,"[web_development, exploratory]","[process, survey]",NaN,[],[{'url': 'https://civic-data.de/leerstandsmeld...,[{'Organizations_id': {'sector': 'development_...,[{'title': 'Unterstützung des Relaunchs des Le...,[],NaN,NaN,NaN


In [8]:
df.to_csv("Correlaid_Projekte_via_API.csv")

# Add license information to API dataset

In [21]:
df["Lizenz"] = "CC-BY 4.0"
df["Lizenz-Organisation"] = "https://correlaid.org/"
df["Kurzzusammenfassung"] = df["translations"].apply(lambda x: x[0]["teaser"])
df["Projektname"] = df["translations"].apply(lambda x: x[0]["title"])
df["Quelle"] = df["project_id"].apply(lambda x: f"https://correlaid.org/daten-nutzen/projektdatenbank/{x}")

def convert_project_status(status):
    if status == "finished":
        return "In Betrieb"
    elif status == "project_work":
        return "In Planung"
    else:   
        return status

df["Status"] = df["project_status"].apply(lambda x: convert_project_status(x))

In [10]:
test = df["Organizations"][6]

for orga in test:
    print(orga["Organizations_id"]["translations"][0]["name"])

Demokratie-Wegweiser
Civic Data Lab


In [11]:


def collect_organisations(orga_information):
    organisation_names = [orga["Organizations_id"]["translations"][0]["name"] for orga in orga_information]
    
    # Check if a certain string is inside the list organisation_names
    if "Civic Data Lab" not in organisation_names:
        organisation_names.append("Civic Data Lab")

    # Convert list organisation_names to string without the square brackets
    organisation_names = ", ".join(organisation_names)

    return organisation_names

df["Organisation"] = df["Organizations"].apply(lambda x: collect_organisations(x))


In [17]:
def collect_project_result_websites(project_outputs):
    project_result_websites = [output["url"] for output in project_outputs]

    # Convert list to string without the square brackets
    project_result_websites = ", ".join(project_result_websites)

    return project_result_websites

df["Webseite-Link"] = df["Projects_Outputs"].apply(lambda x: collect_project_result_websites(x))

# Load traditionally scraped Correlaid data for comparison

In [10]:
correlaid_data_scraping = pd.read_csv(
    "Correlaid_Projekte_enriched.csv",
    sep=";",
    encoding="utf-8")
correlaid_data_scraping

,Quelle,Projektname,Art,Einsatzbereich,Webseite-Link,Organisation,Status,Kurzzusammenfassung,Projekt-Abkürzung,Lizenz,Lizenz-Organisation
0,https://correlaid.org//daten-nutzen/projektdat...,Automatisierung von Reportings aus Evaluations...,"Reporting, Visualisierung, Automatisierte Date...","Bildung, Kinder- und Jugendhilfe, Chancengleic...",https://www.coachatschool.org/,"coach@school e.V., CorrelAid",In Planung,"Das Projekt unterstützt coach@school dabei, Um...",NaN,CC-BY 4.0,https://correlaid.org/
1,https://correlaid.org//daten-nutzen/projektdat...,Improving and upgrading the Silbernetz dashboard,"Dashboard, Reporting, Visualisierung mit Karte...","Senioren, Wohlfahrt, Wirkungsmessung, Transparenz",https://www.silbernetz.org,"CorrelAid, Silbernetz e.V.",In Planung,Dieses Projekt aktualisiert und verbessert das...,NaN,CC-BY 4.0,https://correlaid.org/
2,https://correlaid.org//daten-nutzen/projektdat...,Automatisiertes Qualitätsmanagement für ein Me...,"Prozessautomatisierung, Datenerhebung, Datenan...","Mentoring, Jugendarbeit, Chancengleichheit, Ev...",https://www.sindbad.co.at/,"Sindbad, CorrelAid",In Planung,"Dieses Projekt zielt darauf ab, das interne Qu...",NaN,CC-BY 4.0,https://correlaid.org/
3,https://correlaid.org//daten-nutzen/projektdat...,Data Story mit Konstanzer Klima- und Wetterdaten,"Bericht, Visualisierung, Datenanalyse, Datenan...","Umweltschutz, Nachhaltigkeit, Stadt, Transpare...","https://smart-green-city-konstanz.de/, https:/...","CorrelAid, Smart Green City Konstanz",In Planung,"Das Projekt entwickelt eine Data Story, die Bü...",NaN,CC-BY 4.0,https://correlaid.org/
4,https://correlaid.org//daten-nutzen/projektdat...,Automatisiertes Monitoring der Zielgruppenentw...,"Reporting, Visualisierung, Automatisierung, Wi...","Bildung, Kinder- und Jugendhilfe, Wirkungsmess...",https://chancenwerk.de/,"CorrelAid, Chancenwerk e.V.",In Planung,Das Projekt unterstützt Chancenwerk e.V. dabei...,NaN,CC-BY 4.0,https://correlaid.org/
5,https://correlaid.org//daten-nutzen/projektdat...,Open Data aus Bürger- und Jugendbeteiligungspr...,"Datenerhebung, Datenanwendung für Öffentlichke...","Demokratie, Jugendbeteiligung, Partizipation, ...",https://civic-data.de/kommuki-open-data/,"CorrelAid, Politik zum Anfassen e.V., Civic Da...",Eingestellt,Dieses Projekt machte Open Data aus Bürger- un...,NaN,CC-BY 4.0,https://correlaid.org/
6,https://correlaid.org//daten-nutzen/projektdat...,Automatisierte Fragebogenauswertung mit Genera...,"Generative KI, KI Anwendung, Large Language Mo...","Arbeit mit Kindern, Kinder- und Jugendhilfe, S...",NaN,"Babylotse Frankfurt, Kinderschutzbund Bezirksv...",In Weiterentwicklung,Dieses Projekt nutzte Generative KI und Large ...,NaN,CC-BY 4.0,https://correlaid.org/
7,https://correlaid.org//daten-nutzen/projektdat...,Skalierung der Wirkungsmessung mit KI für In s...,"Analyse von Sensordaten und ML, KI Anwendung, ...","Kinder- und Jugendhilfe, Soziale Arbeit, Wirku...",NaN,"CorrelAid, In safe hands e.V.",In Weiterentwicklung,CorrelAid hat für In safe hands e.V. ein KI-ge...,NaN,CC-BY 4.0,https://correlaid.org/
8,https://correlaid.org//daten-nutzen/projektdat...,Demokratie-Wegweiser,"Visualisierung, Offene Daten, Interaktive Kart...","Demokratie, Partizipation, Transparenz","https://karte.demokratie-wegweiser.de, https:/...","CorrelAid, Civic Data Lab",In Betrieb,Das Projekt Demokratie-Wegweiser zielt darauf ...,NaN,CC-BY 4.0,https://correlaid.org/
9,https://correlaid.org//daten-nutzen/projektdat...,R Wrapper for the Genesis API,"Automatisierte Datenübermittlung, Datenanalyse...","Transparenz, Verwaltung",https://cran.r-project.org/web/packages/restat...,CorrelAid,In Weiterentwicklung,Das R-Paket 'restatis' verbessert und vereinhe...,restatis,CC-BY 4.0,https://correlaid.org/
